# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslanxrz/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [138]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [139]:
import pandas as pd
df = pd.read_csv('/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv')

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### My Lane : Refresh / Content Opurtunity


Why : I am choosing this because it maps to real, recurring editorial decision (what to fix first), it has a clear evaluation metric (Precision@K on a ranked queue), and the starter data shows a large and non-trivial population to work with - over half the pages are trending down and roughly 9300 are both stale and still getting search visibility

In [140]:
# df['trend_direction'].value_counts() / df.shape[0] * 100
df['trend_direction'].value_counts("%") * 100

,proportion
trend_direction,
down,54.206667
stable,19.873333
up,14.626667
new,7.453333
flat,3.840000


 ### **Half the pages are trending down**

In [141]:
len(df[(df['freshness_tier'].isin(['91-180', '181+'])) & (df['avg_position'] > 0)])

9320

### **Roughly 9300 are both stale and still getting search visibility**

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a content editor review first this week?

**Who acts:** The content/SEO editor working through a prioritized queue — they'd spend their limited review hours on the top of my ranked list instead of picking pages arbitrarily or by gut feel.

**Cost of a wrong call:** If the model ranks a fine page as "urgent," the editor wastes review time on it (opportunity cost — a genuinely declining page waits longer). If it misses a real decliner, that page keeps losing visibility unnoticed. Since editor hours are the scarce resource, false positives near the top of the queue are the costliest mistake.

In [142]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


In [143]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
model = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
model.fit(X, y)
print(export_text(model, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



In [144]:
df["decline_score"] = model.predict_proba(X)[:,1]

df.sort_values("decline_score", ascending=False).head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,decline_score
29979,content_6f2f3043b633,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3334.0,20913.0,...,41.9,15.00,7.89,0.00,good,page_3_5,down,-64.6,1,0.611585
29978,content_3dc420aa9809,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2947.0,19594.0,...,7.6,20.00,80.00,0.00,good,page_1,down,-66.5,1,0.611585
29977,content_c87291853cab,client_d029fa3a95,0.0,0.00,LOW,0.00,comparison article,informational,3950.0,27380.0,...,8.2,0.00,33.33,0.00,low,page_1,down,-33.3,1,0.611585
29976,content_851c604b0631,client_6208ef0f77,10.0,0.00,LOW,0.00,keyword article,commercial,5428.0,36179.0,...,16.9,0.00,14.29,0.00,low,striking,down,-45.3,1,0.611585
29975,content_bdee2164f576,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,commercial,6358.0,40791.0,...,25.3,0.00,4.15,0.00,moderate,page_3_5,down,-25.0,1,0.611585
29974,content_b00f10211e25,client_3fdba35f04,70.0,0.99,HIGH,3.24,keyword article,informational,1371.0,8795.0,...,33.2,0.00,13.04,0.00,moderate,page_3_5,down,-56.3,1,0.611585
29972,content_1db0d204d42f,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6862.0,43541.0,...,27.2,1.62,1.86,0.65,good,page_3_5,down,-37.1,1,0.611585
29970,content_4998a1c76243,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,3146.0,20309.0,...,7.1,0.00,0.00,0.00,moderate,page_1,up,90.7,0,0.611585
29969,content_b5db993ce36a,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,commercial,5809.0,38002.0,...,43.7,0.00,2.90,0.00,good,page_3_5,stable,-4.1,0,0.611585
29968,content_179533212cd0,client_f74efabef1,30.0,0.02,LOW,0.00,keyword article,commercial,3884.0,25797.0,...,11.8,17.65,23.53,0.00,moderate,striking,new,NaN,0,0.611585


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [145]:
down = df[df['trend_direction'] == 'down']

print(f"Pages trending down: {len(down):,} ({len(down)/len(df)*100:.1f}% of {len(df):,})")

stale_visible = df[(df['freshness_tier'].isin(['91-180', '181+'])) & (df['avg_position'] > 0)]
print(f"Stale AND still visible in search: {len(stale_visible):,} ({len(stale_visible)/len(df)*100:.1f}%)")

priority_pool = down[(down['search_volume'] > 0) & (down['avg_position'] > 0)]
print(f"Down + real search demand + visible: {len(priority_pool):,}")

Pages trending down: 16,262 (54.2% of 30,000)
Stale AND still visible in search: 9,320 (31.1%)
Down + real search demand + visible: 8,512


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

I can say a page is observed to be declining or stale, and that my ranked queue is directional - a decision support tool that surfaces likely-worthwhile review candidates, not a gaurantee. I will not claim predicts Google's algorithm, and I will not a refresh caused recovery unless I run an actual before/after experiment - correlation between "we refreshed it" and "it went up" is not proof, since I would have no control group

In [146]:
df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


In [147]:
df["freshness_tier"].value_counts()

,count
freshness_tier,
0-30,20480
91-180,9171
31-90,175
181+,174


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.